In [ ]:
# initiale Geocoding
import pickle
import pandas as pd
from geopy.geocoders import Nominatim, Photon
from geopy.extra.rate_limiter import RateLimiter

# --- 1. Pickle laden ---
with open('../../data/processed/06_cleaned_master_data.pkl', "rb") as f:    # ursprünglich 01_cleaned_master_data.pkl
    df = pickle.load(f)

# --- 2. Geocoder einrichten ---
nominatim = Nominatim(user_agent="mein_wetter_projekt_v2")
photon    = Photon(user_agent="mein_wetter_projekt_v2_fallback")

geocode_nominatim = RateLimiter(nominatim.geocode, min_delay_seconds=1)
geocode_photon    = RateLimiter(photon.geocode,    min_delay_seconds=1)

# Bounding Box Europa + Mittelmeer
VIEWBOX = [(-10.0, 60.0), (20.0, 35.0)]


# --- 3. Preprocessing-Funktion: Ortsnamen bereinigen ---
import re

def bereinige_ortsname(ortsname):
    """
    Versucht, aus zusammengesetzten Etappenbezeichnungen den eigentlichen
    Ortsnamen zu extrahieren, den Nominatim kennt.

    Strategien:
    - Klammern entfernen: "Chalet-Reynard (Mont Ventoux)" → "Mont Ventoux"
    - Bindestrich-Kombis aufsplitten: "Galibier Serre-Chevalier" → ["Galibier", "Serre-Chevalier"]
    - Bekannte Präfixe/Suffixe entfernen: "SuperBesse" → "Super Besse"
    """
    kandidaten = [ortsname]  # Original immer als ersten Versuch behalten

    # Inhalt von Klammern als eigenständiger Kandidat
    klammer_inhalt = re.findall(r'\(([^)]+)\)', ortsname)
    kandidaten += klammer_inhalt

    # Version ohne Klammern
    ohne_klammern = re.sub(r'\s*\(.*?\)', '', ortsname).strip()
    if ohne_klammern != ortsname:
        kandidaten.append(ohne_klammern)

    # Leerzeichen-getrennte Teilwörter als Kandidaten (Bsp: "Galibier Serre-Chevalier")
    teile = ortsname.split()
    if len(teile) > 1:
        kandidaten += teile           # Jedes Einzelwort
        kandidaten += [" ".join(teile[-2:]),   # Letzten 2 Wörter
                       " ".join(teile[:2])]    # Ersten 2 Wörter

    # Häufige Zusammenschreibungen normalisieren ("SuperBesse" → "Super Besse")
    normalisiert = re.sub(r'([a-z])([A-Z])', r'\1 \2', ortsname)
    if normalisiert != ortsname:
        kandidaten.append(normalisiert)

    # Duplikate entfernen, Reihenfolge beibehalten
    gesehen = set()
    return [k for k in kandidaten if k and not (k in gesehen or gesehen.add(k))]


# --- 4. Geocoding-Funktion mit automatischem Fallback ---
def get_coords(ortsname):
    """
    Stufenweise Suche:
    1. Nominatim mit Bounding Box (Europa) → verhindert falsche Kontinente
    2. Nominatim ohne Bounding Box (für Israel, Irland, Dänemark etc.)
    3. Photon als zweiter Geocoder-Dienst
    4. Alle obigen Stufen für jeden bereinigten Kandidaten-Namen wiederholen
    """
    kandidaten = bereinige_ortsname(ortsname)

    for kandidat in kandidaten:
        # Stufe 1: Nominatim mit Bounding Box
        try:
            loc = geocode_nominatim(kandidat, viewbox=VIEWBOX, bounded=True)
            if loc:
                return loc.latitude, loc.longitude
        except Exception:
            pass

        # Stufe 2: Nominatim ohne Bounding Box
        try:
            loc = geocode_nominatim(kandidat)
            if loc:
                return loc.latitude, loc.longitude
        except Exception:
            pass

        # Stufe 3: Photon
        try:
            loc = geocode_photon(kandidat)
            if loc:
                return loc.latitude, loc.longitude
        except Exception:
            pass

    return None, None


# --- 5. Koordinaten für alle einzigartigen Orte ermitteln ---
alle_orte = pd.unique(df[["departure", "arrival"]].values.ravel("K"))
coords_cache = {}

for ort in alle_orte:
    if pd.notna(ort):
        coords_cache[ort] = get_coords(ort)
        print(f"{ort}: {coords_cache[ort]}")

# --- 6. Koordinaten in neue Spalten eintragen ---
df["departure_lat"] = df["departure"].map(lambda o: coords_cache.get(o, (None, None))[0])
df["departure_lon"] = df["departure"].map(lambda o: coords_cache.get(o, (None, None))[1])
df["arrival_lat"]   = df["arrival"].map(lambda o: coords_cache.get(o, (None, None))[0])
df["arrival_lon"]   = df["arrival"].map(lambda o: coords_cache.get(o, (None, None))[1])

# --- 7. Verbliebene None-Werte ausgeben ---
print("\n--- Orte ohne Koordinaten (nach allen Fallbacks) ---")
none_orte = [ort for ort, coords in coords_cache.items() if coords == (None, None)]
for ort in none_orte:
    print(f"  ⚠ {ort}")
print(f"\nInsgesamt {len(none_orte)} Orte ohne Koordinaten.")

# --- 8. Speichern ---
df.to_pickle('../../data/processed/cleaned_master_data_with_coordinates.pkl')

print(df[["departure", "departure_lat", "departure_lon",
          "arrival",   "arrival_lat",   "arrival_lon"]].head())

In [ ]:
df1 = pd.read_pickle('../../data/processed/cleaned_master_data_with_coordinates.pkl')

# Einstellung: Alle Zeilen der Series anzeigen (nicht kürzen)
pd.set_option('display.max_rows', None)

# Missing Values berechnen
missing_stats = df1.isnull().sum()

print("Vollständige Liste der Missing Values pro Spalte:")
print("-" * 50)
print(missing_stats)
print("-" * 50)

# Einstellung zurücksetzen (optional, falls du später wieder kurze Prints willst)
pd.reset_option('display.max_rows')

In [ ]:
# initiale Geocoding wird hier nachgebessert
import pickle
import pandas as pd
from geopy.geocoders import Nominatim, Photon
from geopy.extra.rate_limiter import RateLimiter

# --- 1. Pickle laden ---
with open('../data/processed/cleaned_master_data_with_coordinates.pkl', 'rb') as f:
    df = pickle.load(f)

print(f"Datensatz geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
print(df[['departure', 'departure_lat', 'departure_lon',
           'arrival', 'arrival_lat', 'arrival_lon']].head())

# --- 2. Geocoder einrichten ---
geolocator_nom = Nominatim(user_agent="fix_geocoding_v2")
geolocator_pho = Photon(user_agent="fix_geocoding_photon")
geocode_nom = RateLimiter(geolocator_nom.geocode, min_delay_seconds=2, max_retries=3, error_wait_seconds=10)
geocode_pho = RateLimiter(geolocator_pho.geocode, min_delay_seconds=1)

# --- 3. Hilfsfunktionen ---
def is_in_europe_strict(lat, lon):
    """Prüft ob Koordinaten in Westeuropa + Mittelmeerraum liegen."""
    return (36 <= lat <= 71) and (-10 <= lon <= 28)

LAENDER = [
    "France", "Italy", "Spain", "Belgium",
    "Netherlands", "Switzerland", "Germany",
    "Portugal", "Austria", "Denmark", "Ireland"
]

def fix_coords_v2(ortsname):
    """Mehrstufige Neugeocodierung mit Länderzusatz."""
    # Stufe 1: Nominatim mit Länderzusatz (ohne bounded)
    for land in LAENDER:
        try:
            loc = geocode_nom(f"{ortsname}, {land}")
            if loc and is_in_europe_strict(loc.latitude, loc.longitude):
                return loc.latitude, loc.longitude
        except Exception:
            continue

    # Stufe 2: Nominatim ohne Länderzusatz
    try:
        loc = geocode_nom(ortsname)
        if loc and is_in_europe_strict(loc.latitude, loc.longitude):
            return loc.latitude, loc.longitude
    except Exception:
        pass

    # Stufe 3: Photon als Fallback
    try:
        loc = geocode_pho(ortsname)
        if loc and is_in_europe_strict(loc.latitude, loc.longitude):
            return loc.latitude, loc.longitude
    except Exception:
        pass

    return None, None

# --- 4. Fehlerhafte Koordinaten identifizieren ---
mask_invalid_dep = (
    ~df['departure_lat'].between(36, 71) |
    ~df['departure_lon'].between(-10, 28)
)
mask_invalid_arr = (
    ~df['arrival_lat'].between(36, 71) |
    ~df['arrival_lon'].between(-10, 28)
)

falsche_departure = df[mask_invalid_dep][['departure', 'departure_lat', 'departure_lon']].drop_duplicates()
falsche_arrival   = df[mask_invalid_arr][['arrival', 'arrival_lat', 'arrival_lon']].drop_duplicates()

print(f"\nEinzigartige falsche departure-Orte: {len(falsche_departure)}")
print(f"Einzigartige falsche arrival-Orte:   {len(falsche_arrival)}")

# --- 5. Fix-Cache aufbauen ---
fix_cache = {}

print("\n--- Starte Neugeocodierung departure ---")
for _, row in falsche_departure.iterrows():
    ort = row['departure']
    if ort not in fix_cache:
        lat, lon = fix_coords_v2(ort)
        fix_cache[ort] = (lat, lon)
        print(f"  {ort}: ({lat}, {lon})")

print("\n--- Starte Neugeocodierung arrival ---")
for _, row in falsche_arrival.iterrows():
    ort = row['arrival']
    if ort not in fix_cache:
        lat, lon = fix_coords_v2(ort)
        fix_cache[ort] = (lat, lon)
        print(f"  {ort}: ({lat}, {lon})")

# --- 6. Koordinaten im DataFrame aktualisieren ---
print("\n--- Aktualisiere DataFrame ---")
for ort, (lat, lon) in fix_cache.items():
    if lat is not None:
        mask_dep = df['departure'] == ort
        df.loc[mask_dep, 'departure_lat'] = lat
        df.loc[mask_dep, 'departure_lon'] = lon

        mask_arr = df['arrival'] == ort
        df.loc[mask_arr, 'arrival_lat'] = lat
        df.loc[mask_arr, 'arrival_lon'] = lon

# --- 7. Nicht behebbare Orte ausgeben ---
nicht_behebbar = {ort: coords for ort, coords in fix_cache.items() if coords == (None, None)}
print(f"\nOrte ohne gültige Koordinaten (manuell prüfen): {len(nicht_behebbar)}")
for ort in nicht_behebbar:
    print(f"  - {ort}")

# --- 8. Ergebnis prüfen ---
mask_noch_falsch = (
    ~df['departure_lat'].between(36, 71) |
    ~df['departure_lon'].between(-10, 28) |
    ~df['arrival_lat'].between(36, 71) |
    ~df['arrival_lon'].between(-10, 28)
)
print(f"\nNoch fehlerhafte Einträge nach Fix: {mask_noch_falsch.sum()}")

# --- 9. Speichern ---
output_path = '/../../data/processed/cleaned_master_data_coordinates_fixed.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(df, f)

print(f"\nGespeichert unter: {output_path}")

In [ ]:
# Auslandsetappen händisch nachtragen

import pickle

# Pickle laden
with open('../data/processed/cleaned_master_data_coordinates_fixed.pkl', 'rb') as f:
    df = pickle.load(f)

# Deine manuell recherchierten Koordinaten
MANUAL_FIXES = {
    "Base Aerea Rivolto":                              (45.978035,  13.038035),
    "Alto de la Montaña de Cullera":                   (39.184809,  -0.242788),
    "Madrid. Paisaje de la Luz":                       (40.419066,  -3.692132),
    "Alto de Cotobello":                               (43.165263,  -5.681416),
    "Ascensión al Pico Jano. San Miguel de Aguayo":    (43.055960,  -4.023918),
    "Estación de esquí de Ordino Arcalís":             (42.648038,   1.495024),
    "El Ferial Larra Belagua":                         (42.985561,  -0.807107),
    "Revel":                                           (43.463379,   1.991988),
    "Polsa":                                           (45.784176,  10.942843),
    "Alberobello (Pietramadre)":                       (40.787472,  17.238496),
    "Tapia":                                           (42.572478,  -4.073293),
    "ElPozo Alimentación":                             (37.850422,  -1.4255351),
    "Jerusalem":                                       (31.780308, 35.22244453),
    "Budapest":                                        (47.501046, 19.05544648),
    "London":                                          (51.518700, -0.12907561),
    "Monaco":                                          (43.739425, 7.423713845),
    "Rotterdam":                                       (51.910077, 4.476030164),
    "Leeds":                                           (53.803862, -1.54535587),
    "Bilbao":                                          (43.264569, -2.93780558),
    "Parlermo":                                        (38.117488, 13.34690671),
    "Amsterdam":                                       (52.369646, 4.895709733),
    "Bologna":                                         (44.497601, 11.43790919),
   "Torino":                                           (45.072017, 7.687880356),
   "Málaga":                                           (36.717307, -4.43905105),
   "Vigo":                                             (42.545121, -8.72731211),
   "Barcelona":                                        (41.386624, 2.146977364),
   "Milan":                                            (45.375098, 10.41007357)

}

# Koordinaten im DataFrame ersetzen
for ort, (lat, lon) in MANUAL_FIXES.items():
    df.loc[df['departure'] == ort, 'departure_lat'] = lat
    df.loc[df['departure'] == ort, 'departure_lon'] = lon
    df.loc[df['arrival'] == ort,   'arrival_lat']   = lat
    df.loc[df['arrival'] == ort,   'arrival_lon']   = lon
    print(f"  ✓ {ort}: ({lat}, {lon})")

# Prüfen ob noch fehlerhafte Einträge übrig sind
mask_check = (
    ~df['departure_lat'].between(35.5, 72) |
    ~df['departure_lon'].between(-10, 28) |
    ~df['arrival_lat'].between(35.5, 72) |
    ~df['arrival_lon'].between(-10, 28)
)
print(f"\nNoch fehlerhafte Einträge: {mask_check.sum()}")

if mask_check.sum() > 0:
    print(df[mask_check][['departure', 'departure_lat', 'departure_lon',
                           'arrival',   'arrival_lat',   'arrival_lon']].drop_duplicates().to_string())

# Speichern
output_path = '../../data/processed/cleaned_master_data_coordinates_fixed_2.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(df, f)

print(f"\nGespeichert unter: {output_path}")

In [ ]:
# Ausgabe der Koordinaten als Karte (Könnte hier rausgelöscht werden)
import folium
import pandas as pd
import pickle

# Pickle laden
with open('../../data/processed/cleaned_master_data_coordinates_fixed_2.pkl', 'rb') as f:
    df = pickle.load(f)

# Karte zentriert auf Frankreich
m = folium.Map(location=[46.5, 2.5], zoom_start=5)

# Departure-Punkte (blau)
for _, row in df.dropna(subset=['departure_lat', 'departure_lon']).iterrows():
    folium.CircleMarker(
        location=[row['departure_lat'], row['departure_lon']],
        radius=3, color='blue', fill=True,
        popup=row['departure']
    ).add_to(m)

# Arrival-Punkte (rot)
for _, row in df.dropna(subset=['arrival_lat', 'arrival_lon']).iterrows():
    folium.CircleMarker(
        location=[row['arrival_lat'], row['arrival_lon']],
        radius=3, color='red', fill=True,
        popup=row['arrival']
    ).add_to(m)

m.save('../../data/processed/koordinaten_check_2.html')

In [ ]:
#Dieser Code kann gelöscht werden

import pickle

# Pickle laden
with open('../../data/processed/cleaned_master_data_coordinates_fixed_2.pkl', 'rb') as f:
    df = pickle.load(f)

# Deine manuell recherchierten Koordinaten
MANUAL_FIXES = {
    "Base Aerea Rivolto":                              (45.978035,  13.038035),
    "Alto de la Montaña de Cullera":                   (39.184809,  -0.242788),
    "Madrid. Paisaje de la Luz":                       (40.419066,  -3.692132),
    "Alto de Cotobello":                               (43.165263,  -5.681416),
    "Ascensión al Pico Jano. San Miguel de Aguayo":    (43.055960,  -4.023918),
    "Estación de esquí de Ordino Arcalís":             (42.648038,   1.495024),
    "El Ferial Larra Belagua":                         (42.985561,  -0.807107),
    "Revel":                                           (43.463379,   1.991988),
    "Polsa":                                           (45.784176,  10.942843),
    "Alberobello (Pietramadre)":                       (40.787472,  17.238496),
    "Tapia":                                           (42.572478,  -4.073293),
    "ElPozo Alimentación":                             (37.850422,  -1.4255351),
    "Jerusalem":                                       (31.780308, 35.22244453),
    "Budapest":                                        (47.501046, 19.05544648),
    "London":                                          (51.518700, -0.12907561),
    "Monaco":                                          (43.739425, 7.423713845),
    "Rotterdam":                                       (51.910077, 4.476030164),
    "Leeds":                                           (53.803862, -1.54535587),
    "Bilbao":                                          (43.264569, -2.93780558),
    "Parlermo":                                        (38.117488, 13.34690671),
    "Amsterdam":                                       (52.369646, 4.895709733),
    "Bologna":                                         (44.497601, 11.43790919),
   "Torino":                                           (45.072017, 7.687880356),
   "Málaga":                                           (36.717307, -4.43905105),
   "Vigo":                                             (42.545121, -8.72731211),
   "Barcelona":                                        (41.386624, 2.146977364),
   "Milan":                                            (45.375098, 10.41007357)

}

# Koordinaten im DataFrame ersetzen
for ort, (lat, lon) in MANUAL_FIXES.items():
    df.loc[df['departure'] == ort, 'departure_lat'] = lat
    df.loc[df['departure'] == ort, 'departure_lon'] = lon
    df.loc[df['arrival'] == ort,   'arrival_lat']   = lat
    df.loc[df['arrival'] == ort,   'arrival_lon']   = lon
    print(f"  ✓ {ort}: ({lat}, {lon})")

# Prüfen ob noch fehlerhafte Einträge übrig sind
mask_check = (
    ~df['departure_lat'].between(35.5, 72) |
    ~df['departure_lon'].between(-10, 28) |
    ~df['arrival_lat'].between(35.5, 72) |
    ~df['arrival_lon'].between(-10, 28)
)
print(f"\nNoch fehlerhafte Einträge: {mask_check.sum()}")

if mask_check.sum() > 0:
    print(df[mask_check][['departure', 'departure_lat', 'departure_lon',
                           'arrival',   'arrival_lat',   'arrival_lon']].drop_duplicates().to_string())

# Speichern
output_path = '../../data/processed/cleaned_master_data_coordinates_fixed_3.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(df, f)

print(f"\nGespeichert unter: {output_path}")

In [ ]:
#Code kann gelöscht werden
# Ausgabe einer Karte als Übersicht über Koordinaten
import folium
import pandas as pd
import pickle

# Pickle laden
with open('../../data/processed/cleaned_master_data_coordinates_fixed_3.pkl', 'rb') as f:
    df = pickle.load(f)

# Karte zentriert auf Frankreich
m = folium.Map(location=[46.5, 2.5], zoom_start=5)

# Departure-Punkte (blau)
for _, row in df.dropna(subset=['departure_lat', 'departure_lon']).iterrows():
    folium.CircleMarker(
        location=[row['departure_lat'], row['departure_lon']],
        radius=3, color='blue', fill=True,
        popup=row['departure']
    ).add_to(m)

# Arrival-Punkte (rot)
for _, row in df.dropna(subset=['arrival_lat', 'arrival_lon']).iterrows():
    folium.CircleMarker(
        location=[row['arrival_lat'], row['arrival_lon']],
        radius=3, color='red', fill=True,
        popup=row['arrival']
    ).add_to(m)

m.save('../../data/processed/koordinaten_check_3.html')

In [ ]:
import pandas as pd
import numpy as np

# Pickle-Datei einlesen
df0 = pd.read_pickle("../../data/processed/cleaned_master_data_coordinates_fixed_3.pkl")

# Haversine-Funktion zur Berechnung der Entfernung in Kilometern
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Erdradius in km

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

# Entfernung berechnen und als neue Spalte hinzufügen
df0["distance_km_koordinaten"] = haversine(
    df0["departure_lat"],
    df0["departure_lon"],
    df0["arrival_lat"],
    df0["arrival_lon"]
)

# Ergebnis als Pickle-Datei speichern
df0.to_pickle("../../data/processed/cleaned_master_data_coordinates_fixed_3_Koordinatendistanz.pkl")

print("Fertig! Datei gespeichert.")

In [ ]:
import pandas as pd

# Pickle-Datei einlesen
df2 = pd.read_pickle("../../data/processed/cleaned_master_data_coordinates_fixed_3_Koordinatendistanz.pkl")

# Alle nicht-numerischen Zeichen (außer Punkt und Komma) entfernen
df2["distance"] = df2["distance"].astype(str).str.replace(r"[^\d.,]", "", regex=True).str.strip()

# Komma durch Punkt ersetzen (falls europäisches Zahlenformat)
df2["distance"] = df2["distance"].str.replace(",", ".", regex=False)

# In numerischen Typ umwandeln
df2["distance"] = pd.to_numeric(df2["distance"], errors="coerce")

print(df2["distance"].head(20).tolist())
print(f"\nDatentyp: {df2['distance'].dtype}")
print(f"Verbleibende NaN-Werte: {df2['distance'].isna().sum()}")

# Gespeicherte Datei
df2.to_pickle("../../data/processed/cleaned_master_data_coordinates_fixed_3_Koordinatendistanz_ohneKM.pkl")

print("\nDatei erfolgreich gespeichert!")